In [1]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

# ============================================================
# 0. 데이터 불러오기
# ============================================================

# train = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / 'train_split.parquet')
# val = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / 'val_split.parquet')

train = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '2_train_clean2.parquet')
val = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '1_val_fe.parquet')


from sklearn import set_config
set_config(display='text')
# scikit-learn의 HTML 출력 과정에서 발생한 cp949 간 UTF-8 충돌 때문에 넣는 코드

In [2]:
# ============================================================
# 1. 컬럼군 정의
# ============================================================

target_col = 'target_maxrise_6h'  # 모델링 학습부터는 반드시 빼내야 함

# 수위 컬럼군
wl_cols = [
    'wl_t',
    'wl_t_minus_1h',
    'wl_t_minus_3h',
    'wl_t_minus_6h',
    'wl_t_minus_12h',
    'wl_t_minus_24h',
]

# 수위차분(wl_diff) 컬럼군
wl_diff_cols = [
    'wl_t',
    'wl_diff_1h',
    'wl_diff_3h',
    'wl_diff_6h',
    'wl_diff_12h',
    'wl_diff_24h'
]

# 강수 컬럼군
rain_cols = [
    # IMERG강우
    'imerg_rain_1h',
    'imerg_rain_6h',
    'imerg_rain_24h',
    # 지상강우
    'sfc_rain_1h',
    'sfc_rain_6h',
    'sfc_rain_24h',
    # 레이더 반사도
    'radar_reflectivity_mean',
    'radar_reflectivity_p95',
    'radar_reflectivity_ge20_fraction'
]

# 강우강도 컬럼군
rain_int_cols = [
    'imerg_rain_1h',
    'imerg_int_1to6h',
    'imerg_int_6to24h',
    'sfc_rain_1h',
    'sfc_int_1to6h',
    'sfc_int_6to24h'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

num_cols = wl_cols + rain_cols + env_cols
num_cols_fe_wl = num_cols + [
    col for col in wl_diff_cols if col != 'wl_t'
]
num_cols_fe = num_cols_fe_wl + [
    col for col in rain_int_cols if col not in ['imerg_rain_1h', 'sfc_rain_1h']
]

print(f"""<변수개수 검증>
    수위변수: {len(wl_cols)}
    강우변수: {len(rain_cols)}, 
    환경변수: {len(env_cols)}, 
    수위차분(wl_t포함): {len(wl_diff_cols)},
    강우강도(rain_1h포함): {len(rain_int_cols)}
    기존 숫자열(수위 + 강우 + 환경): {len(num_cols)}, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): {len(num_cols_fe_wl)},
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): {len(num_cols_fe)}
""")

<변수개수 검증>
    수위변수: 6
    강우변수: 9, 
    환경변수: 7, 
    수위차분(wl_t포함): 6,
    강우강도(rain_1h포함): 6
    기존 숫자열(수위 + 강우 + 환경): 22, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): 27,
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): 31



In [3]:
# ============================================================
# 2. 종속변수 설정
# ============================================================

y_train = train[target_col]
y_val = val[target_col]

In [4]:
# ============================================================
# 3. 기본모델 만들기(주어진 수치형변수만으로, station_id 제외)
# ============================================================
# train_fe_wl 데이터도 기본모델로는 동일한 결과가 나오는지 검증

# 3-1 기본모델 독립변수 데이터셋 정의

X_train = train[num_cols]
X_val = val[num_cols]

In [5]:
# 3-2. 기본모델 만들기: LightGBM

from lightgbm import LGBMRegressor
from sklearn.metrics import root_mean_squared_error

model_base = LGBMRegressor(
    random_state = 42
)

model_base.fit(
    X_train,
    y_train
)

pred_base = model_base.predict(X_val)

rmse_base = root_mean_squared_error(
    y_val,  # 기준값(정답)
    pred_base
)

print(f"기본모델 RMSE: {rmse_base}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.052561 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5532
[LightGBM] [Info] Number of data points in the train set: 616299, number of used features: 22
[LightGBM] [Info] Start training from score 0.111706
기본모델 RMSE: 0.2152515868067838


In [6]:
# ============================================================
# 4. 모델3: wl_diff(수위차분변수) 포함 모델링
# ============================================================

# 4-1. 모델3 독립변수 데이터셋 정의
X_train_fe_wl = train[num_cols_fe_wl]
X_val_fe_wl = val[num_cols_fe_wl]

# 원핫인코딩

from sklearn.preprocessing import OneHotEncoder

# 원핫인코더 생성
ohe = OneHotEncoder(
    handle_unknown='ignore',  # train에 없던 범주가 나와도 에러 X
    sparse_output=True  # sparse: 희소한, 희소행렬(0이 아닌 값이 드문 행렬)
    # 원핫인코딩 결과를 전체 다 저장 X, 희소한 1의 값의 위치만 저장(메모리 사용 적음)
)

# 학습은 train만
ohe.fit(train[['station_id']])  # sklearn 인코더는 2차원 행렬을 받는다.

# 변환
train_st_id_ohe = ohe.transform(train[['station_id']])
val_st_id_ohe = ohe.transform(val[['station_id']])
# 여기까진 라벨인코더와 동일, 데이터셋에 다시 붙이는게 다름

# train, val 셋에 합치기
from scipy.sparse import hstack
# scipy안에 희소행렬을 다루는 서브캐키지 scipy.sparse
# hstack(horizontal stack)은 concat(axis=1)과 같은 역할, df가 아닌 행렬도 붙일 수 있는 함수

X_train_with_station_fe_wl= hstack([
    X_train_fe_wl,
    train_st_id_ohe
]).tocsr()  # df + 희소행력(ohe) -> csr형태로 저장하여 sparse 상태 유지
# 이 변수는 더이상 df가 아님

X_val_with_station_fe_wl = hstack([
    X_val_fe_wl,
    val_st_id_ohe
]).tocsr()

print("원핫인코딩 전 후 데이터셋 크기 비교: 행은 같고 열만 들어남")
print(X_train_fe_wl.shape, X_train_with_station_fe_wl.shape)
print(X_val_fe_wl.shape, X_val_with_station_fe_wl.shape)
# 제공된 데이터셋의 관측소 개수는 465개(md파일에 명시)
# 기존컬럼 27개 + 465 = 492이 정상

원핫인코딩 전 후 데이터셋 크기 비교: 행은 같고 열만 들어남
(616299, 27) (616299, 492)
(133325, 27) (133325, 492)


In [7]:
# 4-3. wl_diff(수위차분변수) 포함 LightGBM

model_with_wl_diff = LGBMRegressor(
    random_state = 42
)

model_with_wl_diff.fit(
    X_train_with_station_fe_wl,
    y_train  # y_train은 기본모델과 동일
)

pred_with_wl_diff = model_with_wl_diff.predict(X_val_with_station_fe_wl)

rmse_with_wl_diff = root_mean_squared_error(
    y_val,  # y_val도 기본모델과 동일
    pred_with_wl_diff
)

print(f"wl_diff(수위차분변수) 포함 모델 RMSE: {rmse_with_wl_diff}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.084877 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7737
[LightGBM] [Info] Number of data points in the train set: 616299, number of used features: 492
[LightGBM] [Info] Start training from score 0.111706
wl_diff(수위차분변수) 포함 모델 RMSE: 0.20209286473281585


In [8]:
# ============================================================
# 5. 모델4: 강우강도 포함 모델링
# ============================================================

# 5-1. 모델4 독립변수 데이터셋 정의
X_train_fe = train[num_cols_fe]
X_val_fe = val[num_cols_fe]

# 원핫인코딩

from sklearn.preprocessing import OneHotEncoder

# 원핫인코더 생성
ohe = OneHotEncoder(
    handle_unknown='ignore',  # train에 없던 범주가 나와도 에러 X
    sparse_output=True  # sparse: 희소한, 희소행렬(0이 아닌 값이 드문 행렬)
    # 원핫인코딩 결과를 전체 다 저장 X, 희소한 1의 값의 위치만 저장(메모리 사용 적음)
)

# 학습과 변환은 모델3 만들때 했음

# train, val 셋에 합치기
from scipy.sparse import hstack
# scipy안에 희소행렬을 다루는 서브캐키지 scipy.sparse
# hstack(horizontal stack)은 concat(axis=1)과 같은 역할, df가 아닌 행렬도 붙일 수 있는 함수

X_train_with_station_fe = hstack([
    X_train_fe,
    train_st_id_ohe
]).tocsr()  # df + 희소행력(ohe) -> csr형태로 저장하여 sparse 상태 유지
# 이 변수는 더이상 df가 아님

X_val_with_station_fe = hstack([
    X_val_fe,
    val_st_id_ohe
]).tocsr()

print("원핫인코딩 전 후 데이터셋 크기 비교: 행은 같고 열만 들어남")
print(X_train_fe.shape, X_train_with_station_fe.shape)
print(X_val_fe.shape, X_val_with_station_fe.shape)
# 제공된 데이터셋의 관측소 개수는 465개(md파일에 명시)
# 기존컬럼 31개 + 465 = 496이 정상

원핫인코딩 전 후 데이터셋 크기 비교: 행은 같고 열만 들어남
(616299, 31) (616299, 496)
(133325, 31) (133325, 496)


In [9]:
# 5-3. 강우강도 포함 LightGBM

model_with_fe = LGBMRegressor(
    random_state = 42
)

model_with_fe.fit(
    X_train_with_station_fe, 
    y_train
)

pred_with_fe = model_with_fe.predict(X_val_with_station_fe)

rmse_with_fe = root_mean_squared_error(
    y_val, pred_with_fe
)

print(f"강우강도 포함 모델 RMSE: {rmse_with_fe}")
# 오히려 rmse 증가 확인
# 강우강도 데이터셋은 빼고 모델링 진행

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.102548 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8757
[LightGBM] [Info] Number of data points in the train set: 616299, number of used features: 496
[LightGBM] [Info] Start training from score 0.111706
강우강도 포함 모델 RMSE: 0.20213539605544328


In [10]:
# # ============================================================
# # 6. 강우강도변수 제외한 데이터셋 생성
# # ============================================================

# train_fe.drop(
#     [col for col in rain_int_cols
#      if col not in ['imerg_rain_1h', 'sfc_rain_1h']],
#     axis=1
# ).to_parquet(
#     PROJECT_ROOT / 'parqueted_data' / '2_train_wl_diff.parquet'
# )

# val_fe.drop(
#     [col for col in rain_int_cols
#      if col not in ['imerg_rain_1h', 'sfc_rain_1h']],
#     axis=1
# ).to_parquet(
#     PROJECT_ROOT / 'parqueted_data' / '2_val_wl_diff.parquet'
# )

# print(
#     train_fe.drop(
#         [col for col in rain_int_cols
#         if col not in ['imerg_rain_1h', 'sfc_rain_1h']],
#     axis=1
#     ).shape,
#     val_fe.drop(
#         [col for col in rain_int_cols
#         if col not in ['imerg_rain_1h', 'sfc_rain_1h']],
#     axis=1
#     ).shape
# )